# Project 1 RET503 - Deteksi Mobil dan Motor (Google Colab)

Sebelum mulai: menu **Runtime > Change runtime type > T4 GPU**. Jalankan sel dari atas ke bawah.

## 1. Cek GPU dan pasang ultralytics

In [ ]:
!nvidia-smi | head -n 5
!pip install -q ultralytics

## 2. Tulis keempat program ke Colab
Cukup dijalankan, tidak perlu diubah.

In [ ]:
%%writefile 00_siapkan_dataset_yolo.py
"""Mengubah dataset_patrol.zip (folder car/ dan motorcycle/) menjadi dataset_yolo/ untuk program 01-03.

Hasil (struktur sama dengan keluaran 00_ambil_subset_coco.py milik dosen):
    dataset_yolo/
    ├── classes.txt        # car, motorcycle
    ├── images/
    └── labels/

Label:
  - Jika di samping gambar sudah ada file .txt (label YOLO), label itu yang dipakai.
  - Jika belum ada, label dibuat OTOMATIS oleh model YOLO COCO (yolo26n.pt). Label otomatis ini
    HARUS diperiksa dengan 01_cek_anotasi.py dan diperbaiki sebelum hasil dipakai untuk laporan.
  - Gambar yang tidak terdeteksi apa pun tidak dimasukkan; salinannya ada di perlu_label_manual/.

Pemakaian (jalankan dari folder Project1, di samping dataset_patrol.zip):
    python 00_siapkan_dataset_yolo.py            # --timpa untuk membuat ulang dataset_yolo/
Lanjutan  : python 01_cek_anotasi.py dataset_yolo
"""
import argparse
import shutil
import zipfile
from pathlib import Path

ZIP = Path("dataset_patrol.zip")
RAW = Path("raw")
OUT = Path("dataset_yolo")
MANUAL = Path("perlu_label_manual")
KELAS = ["car", "motorcycle"]              # urutan = class_id 0, 1
COCO_ID = {"car": 2, "motorcycle": 3}      # id kelas yang sama di COCO
CONF = 0.25
EXT = {".jpg", ".jpeg", ".png"}

p = argparse.ArgumentParser()
p.add_argument("--timpa", action="store_true", help="hapus dataset_yolo/ lama dan buat ulang")
args = p.parse_args()

if OUT.exists() and not args.timpa:
    raise SystemExit(f"{OUT}/ sudah ada (mungkin sudah kamu perbaiki labelnya), jadi tidak ditimpa.\n"
                     "Hapus foldernya atau jalankan dengan --timpa jika memang ingin membuat ulang.")
if not ZIP.exists() and not RAW.exists():
    raise SystemExit(f"{ZIP} tidak ditemukan. Taruh di folder yang sama dengan skrip ini.")
if not RAW.exists():
    print(f"Mengekstrak {ZIP} ...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(RAW)


def kelas_dari_path(path: Path):
    for bagian in path.relative_to(RAW).parts:
        if bagian.lower() in KELAS:
            return bagian.lower()
    return None


gambar = sorted(f for f in RAW.rglob("*") if f.suffix.lower() in EXT and kelas_dari_path(f))
ada_label_asli = any(f.with_suffix(".txt").exists() for f in gambar)
model = None
if not ada_label_asli:
    from ultralytics import YOLO
    model = YOLO("yolo26n.pt")             # hanya untuk membuat label otomatis

if OUT.exists():
    shutil.rmtree(OUT)
if MANUAL.exists():
    shutil.rmtree(MANUAL)
(OUT / "images").mkdir(parents=True)
(OUT / "labels").mkdir(parents=True)
(OUT / "classes.txt").write_text("\n".join(KELAS) + "\n", encoding="utf-8")

hit_gambar = {k: 0 for k in KELAS}
hit_kotak = {k: 0 for k in KELAS}
dari_asli = otomatis = 0
tanpa_kotak = []
for f in gambar:
    k = kelas_dari_path(f)
    sumber = f.with_suffix(".txt")
    if sumber.exists():
        baris = [b for b in sumber.read_text().splitlines() if b.strip()]
        asli = True
    else:
        r = model.predict(str(f), conf=CONF, verbose=False)[0]
        baris = [f"{KELAS.index(k)} " + " ".join(f"{v:.6f}" for v in box)
                 for box, c in zip(r.boxes.xywhn.tolist(), r.boxes.cls.tolist())
                 if int(c) == COCO_ID[k]]
        asli = False
    nama = f"{k}_{f.stem}"
    if not baris:
        MANUAL.mkdir(exist_ok=True)
        shutil.copy(f, MANUAL / (nama + f.suffix.lower()))
        tanpa_kotak.append(nama + f.suffix.lower())
        continue
    shutil.copy(f, OUT / "images" / (nama + f.suffix.lower()))
    (OUT / "labels" / (nama + ".txt")).write_text("\n".join(baris) + "\n")
    hit_gambar[k] += 1
    hit_kotak[k] += len(baris)
    dari_asli, otomatis = dari_asli + asli, otomatis + (not asli)

(OUT / "CATATAN_LABEL.txt").write_text(
    f"Gambar berlabel asli      : {dari_asli}\n"
    f"Gambar berlabel otomatis  : {otomatis} (model yolo26n.pt/COCO, conf {CONF})\n"
    f"Gambar tanpa kotak        : {len(tanpa_kotak)} (tidak dipakai)\n"
    "Label otomatis hanya memuat objek yang sesuai nama folder gambar (mis. folder motorcycle -> motor saja),\n"
    "sehingga kendaraan lain di latar bisa tidak berlabel. Periksa dengan 01_cek_anotasi.py.\n", encoding="utf-8")

print(f"Kelas           : {', '.join(f'{i} = {k}' for i, k in enumerate(KELAS))}")
print(f"Gambar dipakai  : " + ", ".join(f"{k} {n}" for k, n in hit_gambar.items()))
print(f"Jumlah kotak    : " + ", ".join(f"{k} {n}" for k, n in hit_kotak.items()))
print(f"Label asli / otomatis : {dari_asli} / {otomatis}")
if tanpa_kotak:
    print(f"{len(tanpa_kotak)} gambar tanpa kotak -> salinan di {MANUAL}/ (beri label manual jika mau dipakai)")
for sisa in ("dataset_v1", "hasil_praktik"):
    if Path(sisa).exists():
        print(f"[!] Folder lama '{sisa}/' masih ada. Hapus dulu sebelum menjalankan 02 dan 03 agar tidak bercampur.")
if otomatis:
    print("\nPERHATIAN: sebagian label dibuat otomatis. Periksa dan perbaiki dulu sebelum dipakai untuk laporan.")
print(f"Siap di {OUT}/ -> lanjutkan dengan: python 01_cek_anotasi.py {OUT}")

In [ ]:
%%writefile 01_cek_anotasi.py
"""Memeriksa kualitas anotasi format YOLO dan menggambar kotaknya untuk dicek visual.

Struktur folder yang diharapkan:
    dataset_yolo/
    ├── classes.txt          # satu nama kelas per baris, urutan = class_id
    ├── images/  *.jpg / *.png
    └── labels/  *.txt       # nama sama dengan gambar

Pemakaian : python 01_cek_anotasi.py [folder_dataset]
Hasil     : laporan di layar + folder cek_visual/ berisi gambar dengan kotak
"""
import sys
from collections import Counter
from pathlib import Path

import cv2

ROOT = Path(sys.argv[1] if len(sys.argv) > 1 else "dataset_yolo")
if not (ROOT / "classes.txt").exists():
    sys.exit(f"Folder '{ROOT}' atau file '{ROOT / 'classes.txt'}' belum ada.\n"
             "Siapkan dataset dulu: ekspor dari CVAT, atau jalankan 00_siapkan_dataset_yolo.py (untuk dataset_patrol) /\n"
             "00_siapkan_dari_voc.py untuk dataset publik.")
KELAS = [k.strip() for k in (ROOT / "classes.txt").read_text(encoding="utf-8").splitlines() if k.strip()]
MIN_SISI_PX = 8          # kotak lebih kecil dari ini patut dicurigai
WARNA = [(0, 200, 0), (0, 140, 255), (255, 80, 0), (200, 0, 200), (0, 200, 200)]

gambar = sorted(p for p in (ROOT / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
out_dir = ROOT / "cek_visual"
out_dir.mkdir(exist_ok=True)
hitung, masalah = Counter(), []
tanpa_label, kosong = 0, 0

for img_path in gambar:
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    lbl = ROOT / "labels" / (img_path.stem + ".txt")
    if not lbl.exists():
        tanpa_label += 1
        masalah.append(f"{img_path.name}: tidak ada file label")
        continue
    baris = [b for b in lbl.read_text().splitlines() if b.strip()]
    if not baris:
        kosong += 1                      # boleh: gambar latar tanpa objek
    for i, b in enumerate(baris, 1):
        v = b.split()
        if len(v) != 5:
            masalah.append(f"{lbl.name} baris {i}: harus 5 angka, ada {len(v)}")
            continue
        cid, cx, cy, bw, bh = int(v[0]), *map(float, v[1:])
        if not 0 <= cid < len(KELAS):
            masalah.append(f"{lbl.name} baris {i}: class_id {cid} tidak ada di classes.txt")
            continue
        if not all(0 <= t <= 1 for t in (cx, cy, bw, bh)):
            masalah.append(f"{lbl.name} baris {i}: nilai di luar 0-1 (belum dinormalisasi?)")
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        if bw * w < MIN_SISI_PX or bh * h < MIN_SISI_PX:
            masalah.append(f"{lbl.name} baris {i}: kotak sangat kecil ({bw * w:.0f}x{bh * h:.0f} px)")
        hitung[KELAS[cid]] += 1
        warna = WARNA[cid % len(WARNA)]
        cv2.rectangle(img, (x1, y1), (x2, y2), warna, 2)
        cv2.putText(img, KELAS[cid], (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, warna, 2)
    cv2.imwrite(str(out_dir / img_path.name), img)

print(f"Gambar          : {len(gambar)}")
print(f"Tanpa file label: {tanpa_label}")
print(f"Label kosong    : {kosong} (gambar latar tanpa objek)")
print("Jumlah kotak per kelas:")
for k in KELAS:
    print(f"  {k:10s}: {hitung[k]}")
print(f"\n{len(masalah)} masalah ditemukan" + (":" if masalah else "."))
for m in masalah[:20]:
    print("  [!]", m)
print(f"\nGambar dengan kotak tersimpan di {out_dir}/ untuk dicek visual.")

In [ ]:
%%writefile 02_split_dataset_yolo.py
"""Membagi dataset YOLO menjadi train/val/test dan membuat data.yaml untuk Ultralytics.

Masukan  : dataset_yolo/ (classes.txt, images/, labels/) hasil anotasi
Keluaran : dataset_v1/
           ├── data.yaml
           ├── images/{train,val,test}/
           └── labels/{train,val,test}/

Pemakaian : python 02_split_dataset_yolo.py [folder_masukan] [folder_keluaran]
Jalankan SETELAH 01_cek_anotasi.py tidak menemukan masalah.
"""
import random
import shutil
import sys
from pathlib import Path

SRC = Path(sys.argv[1] if len(sys.argv) > 1 else "dataset_yolo")
DST = Path(sys.argv[2] if len(sys.argv) > 2 else "dataset_v1")
PROPORSI = {"train": 0.7, "val": 0.2, "test": 0.1}
random.seed(42)                      # pembagian bisa diulang dengan hasil sama

kelas = [k.strip() for k in (SRC / "classes.txt").read_text().splitlines() if k.strip()]
gambar = sorted(p for p in (SRC / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
gambar = [g for g in gambar if (SRC / "labels" / f"{g.stem}.txt").exists()]   # hanya yang berlabel
random.shuffle(gambar)

n = len(gambar)
n_train = int(n * PROPORSI["train"])
n_val = int(n * PROPORSI["val"])
bagian = {"train": gambar[:n_train], "val": gambar[n_train:n_train + n_val], "test": gambar[n_train + n_val:]}

for nama, daftar in bagian.items():
    (DST / "images" / nama).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / nama).mkdir(parents=True, exist_ok=True)
    for g in daftar:
        shutil.copy(g, DST / "images" / nama / g.name)
        shutil.copy(SRC / "labels" / f"{g.stem}.txt", DST / "labels" / nama / f"{g.stem}.txt")
    print(f"{nama:5s}: {len(daftar)} gambar")

yaml = [f"path: {DST.resolve().as_posix()}", "train: images/train", "val: images/val",
        "test: images/test", "names:"] + [f"  {i}: {k}" for i, k in enumerate(kelas)]
(DST / "data.yaml").write_text("\n".join(yaml) + "\n")
print(f"\ndata.yaml dibuat di {DST / 'data.yaml'}")

In [ ]:
%%writefile 03_latih_bandingkan.py
"""Praktik terpadu minggu 4-7, tahap 3-5: melatih, tuning, memilih, dan menguji model YOLO.

Eksperimen yang dijalankan (pengaturan lain SAMA untuk semua):
  E1 pretrained : model pretrained COCO (transfer learning)      -> baseline
  E2 scratch    : arsitektur sama, bobot acak (tanpa transfer)   -> pembanding
  E3 tuning     : E1 + SATU perubahan (default imgsz=640)        -> eksperimen tuning
Model terbaik dipilih dengan mAP50-95 pada data VAL, lalu diuji SEKALI pada data TEST.

Pemakaian:
  python 03_latih_bandingkan.py --nim 4222101001 --nama "Budi Santoso"
  python 03_latih_bandingkan.py --nim ... --nama "..." --epochs 30 --ubah epochs=60
  python 03_latih_bandingkan.py --nim ... --nama "..." --model yolov8n   (jika YOLO26 belum tersedia)

Membutuhkan: pip install ultralytics, dan dataset_v1/data.yaml dari 02_split_dataset_yolo.py.
Disarankan memakai GPU (komputer lab atau Google Colab).
"""
import argparse
import csv
import getpass
import hashlib
import platform
import time
from datetime import datetime
from pathlib import Path

from ultralytics import YOLO

p = argparse.ArgumentParser()
p.add_argument("--nim", required=True)
p.add_argument("--nama", required=True)
p.add_argument("--data", default="dataset_v1/data.yaml")
p.add_argument("--model", default="yolo26n", help="nama model tanpa ekstensi, misal yolo26n atau yolov8n")
p.add_argument("--epochs", type=int, default=30)
p.add_argument("--imgsz", type=int, default=416)
p.add_argument("--batch", type=int, default=16)
p.add_argument("--device", default=None, help="0 untuk GPU pertama, cpu untuk CPU (default: otomatis)")
p.add_argument("--ubah", default="imgsz=640", help="SATU perubahan untuk E3, format kunci=nilai")
args = p.parse_args()

# --- Satu perubahan untuk eksperimen tuning (E3) ---
kunci, nilai = args.ubah.split("=", 1)
try:
    nilai = int(nilai)
except ValueError:
    try:
        nilai = float(nilai)
    except ValueError:
        pass                                   # biarkan sebagai teks, misal optimizer=SGD

DATA = Path(args.data)
if not DATA.exists():
    raise SystemExit(f"{DATA} tidak ditemukan. Jalankan dulu 02_split_dataset_yolo.py.")
PROJECT = Path("hasil_praktik").resolve()


def sidik_jari_dataset(root: Path) -> str:
    """Hash isi semua file label: dataset sama -> kode sama (untuk mendeteksi salin-menyalin)."""
    h = hashlib.sha256()
    for f in sorted((root / "labels").rglob("*.txt")):
        h.update(f.read_bytes())
    return h.hexdigest()[:12].upper()


def latih(nama, model_file, pretrained, **ubah):
    """Melatih satu eksperimen, lalu mengevaluasi best.pt di data val."""
    cfg = dict(data=str(DATA), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch,
               seed=0, deterministic=True, project=str(PROJECT), name=nama,
               exist_ok=True, pretrained=pretrained, plots=True, verbose=False)
    if args.device is not None:
        cfg["device"] = args.device
    cfg.update(ubah)
    print(f"\n=== {nama}: {model_file} | pretrained={pretrained} | {ubah or 'pengaturan dasar'} ===")
    model = YOLO(model_file)
    t0 = time.time()
    model.train(**cfg)
    menit = (time.time() - t0) / 60
    best = Path(model.trainer.save_dir) / "weights" / "best.pt"
    m = YOLO(str(best)).val(data=str(DATA), split="val", imgsz=cfg["imgsz"], batch=args.batch,
                            plots=False, verbose=False)
    return {"eksperimen": nama, "perubahan": ", ".join(f"{k}={v}" for k, v in ubah.items()) or "-",
            "pretrained": pretrained, "mAP50": round(m.box.map50, 4), "mAP50-95": round(m.box.map, 4),
            "precision": round(m.box.mp, 4), "recall": round(m.box.mr, 4),
            "inferensi_ms": round(m.speed["inference"], 2), "waktu_latih_menit": round(menit, 2),
            "imgsz": cfg["imgsz"], "bobot": str(best)}


hasil = [
    latih("E1_pretrained", f"{args.model}.pt", True),
    latih("E2_scratch", f"{args.model}.yaml", False),
    latih("E3_tuning", f"{args.model}.pt", True, **{kunci: nilai}),
]

# --- Pilih model terbaik dengan data VAL, lalu uji SEKALI di data TEST ---
terbaik = max(hasil, key=lambda r: r["mAP50-95"])
uji = YOLO(terbaik["bobot"]).val(data=str(DATA), split="test", imgsz=terbaik["imgsz"],
                                  batch=args.batch, plots=False, verbose=False)

garis = "=" * 78
baris = [garis, "  PRAKTIK TERPADU MINGGU 4-7 - RET503", garis,
         f"  NIM / Nama    : {args.nim} / {args.nama}",
         f"  Waktu         : {datetime.now():%d-%m-%Y %H:%M:%S}",
         f"  Komputer/user : {platform.node()} / {getpass.getuser()}",
         f"  Dataset       : {DATA.parent}  (sidik jari label: {sidik_jari_dataset(DATA.parent)})",
         f"  Pengaturan    : model {args.model}, epochs {args.epochs}, imgsz {args.imgsz}, batch {args.batch}",
         garis,
         f"  {'Eksperimen':15s}{'Perubahan':14s}{'mAP50':>8s}{'mAP50-95':>10s}{'P':>7s}{'R':>7s}{'ms':>7s}{'menit':>8s}"]
for r in hasil:
    baris.append(f"  {r['eksperimen']:15s}{r['perubahan']:14s}{r['mAP50']:8.3f}{r['mAP50-95']:10.3f}"
                 f"{r['precision']:7.3f}{r['recall']:7.3f}{r['inferensi_ms']:7.1f}{r['waktu_latih_menit']:8.1f}")
baris += [garis,
          f"  Model terpilih (mAP50-95 val tertinggi): {terbaik['eksperimen']}",
          f"  Hasil di data TEST (sekali saja)       : mAP50 {uji.box.map50:.3f} | mAP50-95 {uji.box.map:.3f}",
          garis]
laporan = "\n".join(baris)
print("\n" + laporan)

Path(f"hasil_praktik_{args.nim}.txt").write_text(laporan + "\n", encoding="utf-8")
with open(f"hasil_praktik_{args.nim}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(hasil[0].keys()))
    w.writeheader()
    w.writerows(hasil)
print(f"Tersimpan: hasil_praktik_{args.nim}.txt dan .csv | grafik tiap eksperimen di {PROJECT}/")

## 3. Unggah dataset_patrol.zip
Pilih file `dataset_patrol.zip` dari laptop.

In [ ]:
from google.colab import files
files.upload()

## 4. Program 00: buat dataset_yolo/ (label otomatis)

In [ ]:
!python 00_siapkan_dataset_yolo.py

## 5. Program 01: cek anotasi
Lihat laporan masalahnya, lalu gambar contoh di bawahnya.

In [ ]:
!python 01_cek_anotasi.py dataset_yolo

In [ ]:
import glob, os
import matplotlib.pyplot as plt, matplotlib.image as mpimg
fs = sorted(glob.glob('dataset_yolo/cek_visual/*'))[:12]
fig, ax = plt.subplots(3, 4, figsize=(14, 9))
for a in ax.ravel(): a.axis('off')
for a, f in zip(ax.ravel(), fs):
    a.imshow(mpimg.imread(f)); a.set_title(os.path.basename(f)[:22], fontsize=8)
plt.show()

### 5b. (Disarankan) Perbaiki label
Label otomatis harus diperiksa. Unduh dataset, perbaiki label di CVAT / LabelImg / Roboflow (ekspor format **YOLO**), lalu unggah kembali sebagai zip berisi folder `dataset_yolo/` (isi: `classes.txt`, `images/`, `labels/`). Lewati 5b dan 5c jika tidak ada koreksi.

In [ ]:
!zip -qr dataset_yolo_otomatis.zip dataset_yolo -x 'dataset_yolo/cek_visual/*'
files.download('dataset_yolo_otomatis.zip')

### 5c. Unggah dataset_yolo yang sudah dikoreksi (opsional)
Folder `dataset_yolo/` yang lama akan diganti.

In [ ]:
import shutil
up = files.upload()
shutil.rmtree('dataset_yolo', ignore_errors=True)
shutil.unpack_archive(list(up.keys())[0], '.')
!python 01_cek_anotasi.py dataset_yolo

## 6. Program 02: bagi data train / val / test

In [ ]:
!rm -rf dataset_v1
!python 02_split_dataset_yolo.py

## 7. Program 03: latih E1, E2, E3
Ganti NIM dan nama dulu. Butuh waktu beberapa menit.

In [ ]:
!python 03_latih_bandingkan.py --nim 4222401012 --nama "Nur Hafidzatun Nabila"

## 8. Unduh hasil untuk dikumpulkan

In [ ]:
!zip -qr hasil_untuk_github.zip hasil_praktik hasil_praktik_*.txt hasil_praktik_*.csv dataset_yolo dataset_v1 -x '*.pt'
files.download('hasil_untuk_github.zip')

File `.pt` tidak ikut agar zip kecil. Jika perlu, unduh dari `hasil_praktik/E1_pretrained/weights/best.pt`.